# 05-1. 구간별 추정과 결빙 영향인자

구간별 추정 관측값으로 결빙 영향인자와 임계값을 도출하고, 위험도 기준표와 지도를 만든다(05). 이 노트북은 1~2장이다.

05 구성: **05-1 구간별 추정·결빙 영향인자(1~2장)** · 05-2 위험도 기준표·산출·지도(3~6장)

1. 03·04 모델로 구간(1km)별 노면온도와 결빙관련 노면 확률을 1시간 간격으로 추정한다.
2. 학습 지점 관측자료로 결빙 발생 영향인자와 환경조건(임계값)을 정리한다.

- 입력: 02-2의 구간·지점 변수, 03-3의 노면온도 추정 모델과 03-2의 구간 B3 추정값, 04-1의 결빙 분류 모델,
  02-3의 ERA5-Land (`era5_10min.parquet`)
- 출력: 구간별 추정값·노면온도 요약·결빙 가능 조건 (`data/processed/`, 05-2에서 사용),
  결빙 영향인자·환경조건 표 (`data/results/icing_factors.csv`, `icing_conditions.csv`)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]  # notebooks/<단계 폴더>/ 에서 실행
sys.path.insert(0, str(ROOT))

from src import config

In [2]:
import json
import warnings

import geopandas as gpd
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
N_WINTERS = len(config.PERIODS)

In [3]:
segments = gpd.read_file(config.PROCESSED_DIR / "segments.gpkg")
seg_feat = pd.read_parquet(config.PROCESSED_DIR / "segment_features.parquet").merge(
    pd.read_parquet(config.PROCESSED_DIR / "segment_b3.parquet"), on=["seg_id", "time"], how="left",
    validate="one_to_one")  # 03-2의 구간 B3 (밑그림 + 그날의 오차 크리깅)
stations = pd.read_parquet(config.PROCESSED_DIR / "stations.parquet").set_index("stn")
station_seg = pd.read_csv(config.PROCESSED_DIR / "station_segment_match.csv")
sf = pd.read_parquet(config.PROCESSED_DIR / "station_features.parquet").merge(
    pd.read_parquet(config.PROCESSED_DIR / "station_estimates.parquet"), on=["stn", "time"])
est_meta = json.load(open(config.MODEL_DIR / "road_temp_meta.json", encoding="utf-8"))
cls_meta = json.load(open(config.MODEL_DIR / "icing_meta.json", encoding="utf-8"))
print(f"구간 {len(segments)}개, 구간 시계열 {seg_feat.shape[0]:,}행 (1시간 간격)")

구간 233개, 구간 시계열 1,342,080행 (1시간 간격)


## 1. 구간별 관측값 추정

03의 노면온도 추정 앙상블(LightGBM + Ridge + 주변 RWIS IDW + B3 밑그림·크리깅)과 04의 결빙 분류 모델을 구간 중간점 변수에 적용한다.

In [4]:
def predict_members(df):
    """03의 최종 추정 방법(앙상블)의 구성원별 노면온도 추정값."""
    feats = est_meta["features"]
    booster = lgb.Booster(model_file=str(config.MODEL_DIR / "road_temp_lgbm.txt"))
    ridge = joblib.load(config.MODEL_DIR / "road_temp_ridge.joblib")
    return pd.DataFrame({"LightGBM(핵심)": booster.predict(df[feats]), "Ridge(핵심)": ridge.predict(df[feats]),
                         "B1 주변 노면온도": df["nbr_rst"].to_numpy(), est_meta["b3"]: df["b3"].to_numpy()}, index=df.index)


def predict_icing(df):
    """04의 최종 결빙 분류 모델로 결빙(결빙관련 노면) 확률을 구한다."""
    path = config.MODEL_DIR / cls_meta["model_file"]
    if cls_meta["model_type"] == "lightgbm":
        return lgb.Booster(model_file=str(path)).predict(df[cls_meta["features"]])
    return joblib.load(path).predict_proba(df[cls_meta["features"]])[:, 1]


members = predict_members(seg_feat)
seg_feat["rst_est"] = members[est_meta["ensemble"]].mean(axis=1).astype("float32")
seg_feat["rst_dpd"] = seg_feat["rst_est"] - seg_feat["td"]
seg_feat["p_icing"] = predict_icing(seg_feat).astype("float32")
seg_feat[["rst_est", "p_icing"]].describe().round(3)

,rst_est,p_icing
count,1342080.000,1342080.000
mean,4.498,0.024
std,6.518,0.112
min,-13.017,0.000
25%,-0.339,0.000
50%,3.742,0.001
75%,8.642,0.002
max,27.742,0.981


구간 추정이 관측과 맞는지, RWIS 관측소가 있는 구간에서 정시 관측값과 비교한다.
검증 지점 3곳은 학습에 쓰지 않았으므로 독립적인 확인이 된다.

In [5]:
obs = sf[sf["time"].dt.minute == 0][["stn", "time", "rst"]].merge(station_seg[["stn", "seg_id", "holdout"]], on="stn")
chk = obs.merge(seg_feat[["seg_id", "time", "rst_est"]], on=["seg_id", "time"]).dropna()
err = chk["rst_est"] - chk["rst"]
seg_check = (chk.assign(err=err).groupby(["stn", "seg_id", "holdout"])["err"]
             .agg(BIAS="mean", MAE=lambda e: e.abs().mean(), RMSE=lambda e: np.sqrt((e ** 2).mean())))
seg_check = seg_check.join(stations["name"], on="stn")
seg_check.round(2)

,,,BIAS,MAE,RMSE,name
stn,seg_id,holdout,,,,
12103,SH-099,False,0.18,0.45,0.67,군산휴(도)
12209,SH-070,False,0.73,0.80,1.16,서김제(도)
12210,SH-048,False,0.03,0.37,0.48,부안고려청자휴(도)
12211,SH-021,False,0.41,0.49,0.62,고인돌휴(도)
12409,SH-091,False,0.70,0.72,0.88,우곡3교(도)
12410,SH-044,True,-0.54,0.68,0.91,부곡교(도)
23202,HN-083,False,-0.33,0.39,0.51,여산휴(도)
23203,HN-067,False,-0.06,0.34,0.47,삼례졸음(도)
23204,HN-020,False,0.04,0.35,0.48,정읍휴(도)


학습 지점의 구간 추정은 자기 관측이 주변 RWIS 변수에 들어가 있어 오차가 작게 나온다.
검증 지점 구간(holdout=True)의 오차가 03-3의 검증 지점 성능과 비슷하면 구간 추정이 일관되게 동작하는 것이다.
검증 지점 구간은 부곡교 0.91, 해전 1.00, 삼봉 0.77°C로 03-3과 같다.

학습 지점 구간은 B3를 더하기 전(0.32~0.70°C)보다 오차가 커졌다(0.37~1.16°C). B3는 원천 관측소의 지문을 옮기지 않고(φ = 0)
그날의 오차도 너깃만큼 걸러서 옮기므로, 관측소 바로 옆 구간에서도 그 관측값을 그대로 따라가지 않기 때문이다.
휴게소 안 센서처럼 본선과 다른 자리의 버릇은 옮기지 않는 편이 맞지만, 서김제(0.70 → 1.16°C)·우곡3교(0.54 → 0.88°C)처럼
본선 위 센서가 있는 구간에서는 추정이 그 관측값에서 멀어진다.

구간 추정에서 유의할 점 (03-2의 3-4~3-6):

- 추정 오차의 절반 가까이는 지점마다 같은 시각에 반복되는 고유 편향이다. 특히 한낮에 크다.
- 학습 지점 14곳 중 7곳이 휴게소·졸음쉼터 안의 센서이다. 이 센서들은 본선 센서보다 한낮에 약 1.8°C, 밤에 0.3~0.4°C 따뜻하게 관측된다.
  이 차이를 보정해 보았지만 검증 지점에서 오히려 나빠져서 적용하지 않았다.
- 결빙 위험은 주로 밤·새벽에 판단하고, 위험도는 구간 간 상대 비교로 쓴다. 그래서 이 편향의 영향은 한낮보다 작다.
- 편향의 방향도 일정하지 않다. 학습 지점의 본선 센서에서는 추정이 실제보다 높은 경향이 있었다.
  반면 검증 지점 3곳(모두 본선)에서는 추정이 평균 0.4°C 낮았다.
- B3는 기온 보간의 감률(0.65°C/100m)을 그대로 써서, 주변 관측소보다 높은 곳의 밤 노면온도를 조금 낮게 추정하는 경향이 있다 (03-2의 3-6-6).
  산지 구간(통영-대전중부선)의 밤 저온 노출은 조금 크게 나올 수 있다.

B3를 더해 구간 추정이 얼마나 바뀌었는지 기존 앙상블(LightGBM + Ridge + B1)과 비교한다 (값은 새 앙상블 - 기존 앙상블). B1은 가까운 관측소의 노면온도를 지문(그 자리의 버릇)까지 그대로 옮기고, B3는 밑그림 위에 그날의 오차만 옮긴다 (03-2의 3-6). 차이를 가장 가까운 학습 관측소까지 거리별로 본다.

In [6]:
_g = gpd.GeoSeries(gpd.points_from_xy(stations["lon"], stations["lat"]), crs=4326).to_crs(5179)
train_xy = np.c_[_g.x, _g.y][~stations["holdout"].to_numpy()]
seg_near = pd.Series(cdist(segments[["mid_x", "mid_y"]].to_numpy(), train_xy).min(axis=1) / 1000,
                     index=segments["seg_id"])  # 가장 가까운 학습 관측소까지 거리 (km)
diff = seg_feat["rst_est"] - members[["LightGBM(핵심)", "Ridge(핵심)", "B1 주변 노면온도"]].mean(axis=1)
by_seg = pd.DataFrame({"평균 차이": diff.groupby(seg_feat["seg_id"]).mean(),
                       "평균 |차이|": diff.abs().groupby(seg_feat["seg_id"]).mean(),
                       "밤 평균 차이": diff[seg_feat["sun_elev"] < 0].groupby(seg_feat["seg_id"]).mean(),
                       "한낮 평균 차이": diff[seg_feat["time"].dt.hour.between(11, 14)].groupby(seg_feat["seg_id"]).mean()})
band = pd.cut(seg_near.reindex(by_seg.index), [0, 2, 5, 10, 20], labels=["2km 이내", "2~5km", "5~10km", "10~20km"])
b3_effect = by_seg.groupby(band, observed=True).mean()
b3_effect.insert(0, "구간 수", band.value_counts())
b3_effect.round(2)

,구간 수,평균 차이,평균 |차이|,밤 평균 차이,한낮 평균 차이
2km 이내,58,0.01,0.21,-0.00,0.06
2~5km,73,0.00,0.20,-0.02,0.07
5~10km,63,0.07,0.16,0.05,0.16
10~20km,39,0.12,0.15,0.07,0.27


- 바뀐 크기(평균 |차이|)는 0.15~0.21°C로 작고, 관측소에 가까운 구간에서 오히려 조금 크다. 가까운 구간일수록 B1이 그 관측소의 버릇을
  많이 옮기는데, B3는 이를 옮기지 않기 때문이다.
- 관측소에서 먼 구간(10~20km)은 평균 0.12°C, 한낮 0.27°C 따뜻해졌다. 먼 구간에서 B1은 멀리 있는 관측소 세 곳의 노면온도를 고도만 보정해 옮기지만,
  B3는 그 구간의 일사·지형 그늘·운량으로 밑그림을 계산하므로 차이가 한낮에 커진다.
- 결빙 판단에 중요한 밤의 변화는 모든 거리대에서 0.1°C 미만이다. 그래서 05-2의 위험도 등급별 구간 수는 거의 같다 (관심 58 → 59, 경계 51 → 50).

In [7]:
night = seg_feat["sun_elev"] < 0
seg_summary = seg_feat.groupby("seg_id").agg(
    rst_mean=("rst_est", "mean"),
    rst_min=("rst_est", "min"),
).join(seg_feat[night].groupby("seg_id")["rst_est"].mean().rename("rst_night_mean"))
seg_summary["rst_le0_hours"] = seg_feat.assign(c=seg_feat["rst_est"] <= 0).groupby("seg_id")["c"].sum() / N_WINTERS
seg_summary.round(2).describe()

,rst_mean,rst_min,rst_night_mean,rst_le0_hours
count,233.000000,233.000000,233.000000,233.000000
mean,4.498197,-10.499056,2.179614,773.667382
std,0.751830,0.695641,0.823552,131.145237
min,0.690000,-13.020000,-1.780000,660.000000
25%,4.260000,-10.790000,2.180000,694.000000
50%,4.810000,-10.600000,2.530000,721.000000
75%,4.990000,-10.150000,2.700000,774.500000
max,5.260000,-9.250000,2.830000,1476.500000


## 2. 결빙 영향인자 및 환경조건(임계값)

학습 지점 14곳의 10분 관측자료로 Ice가 관측된 때와 그렇지 않은 때의 조건을 비교한다 (검증 지점 제외).
노면온도는 실측값, 기상은 02-2의 보간값을 쓴다.
추가 자료로 ERA5-Land(02-3)의 지난 3시간 강수·강설과 강설 비율도 함께 본다. ERA5는 약 9km 격자 값이라 지점의 실제 강수와 다를 수 있다.

이론적으로 결빙은 **노면 위의 물이 어는점 이하의 노면에서 얼어** 생기고, 물은 강수·녹은 눈·수증기 응결(서리) 세 경로로 공급된다
(Zerr 1997, Hong et al. 2022에서 재인용). 01-6의 13장에서 확인한 것처럼 이 지역 Ice의 81%는 강수 중이거나 강수 직후이고,
노면센서는 노면 1.5°C 미만에서만 결빙 계열 상태를 판정한다. 그래서 환경조건은 **저온(필요조건) + 수분 공급(결정 요인)**으로 구성한다.

In [8]:
tr = sf[~sf["stn"].isin(config.HOLDOUT_STATIONS) & sf["ice"].notna()].copy()
tr["rst_td"] = tr["rst"] - tr["td"]
era5 = pd.read_parquet(config.PROCESSED_DIR / "era5_10min.parquet",
                       columns=["stn", "time", "sf_mmh", "tp3h", "sf3h", "snowfrac3h"])
e5 = tr[["stn", "time"]].merge(era5, on=["stn", "time"], how="left", validate="one_to_one")
for c in ["sf_mmh", "tp3h", "sf3h", "snowfrac3h"]:
    tr[f"era5_{c}"] = e5[c].to_numpy()  # 02-3의 ERA5-Land (약 9km 격자)
FACTORS = {
    "rst": "노면온도(°C)", "ta": "기온(°C)", "ta_min12h": "12시간 최저기온(°C)", "td": "이슬점(°C)",
    "rh": "상대습도(%)", "rst_td": "노면온도-이슬점(°C)", "ws": "풍속(m/s)", "cloud": "전운량(1/10)",
    "rain_hrs12h": "최근 12시간 강수시간(h)", "hrs_since_rain": "강수 후 경과시간(h)", "snow": "적설(cm)",
    "sun_elev": "태양고도(°)",
    "era5_tp3h": "ERA5 3시간 강수(mm)", "era5_sf3h": "ERA5 3시간 강설(mm, 물 환산)", "era5_snowfrac3h": "ERA5 강설 비율(3시간)",
}
ice, non = tr[tr["ice"] == 1], tr[tr["ice"] == 0]
cond_table = pd.DataFrame({
    "변수": FACTORS.values(),
    "Ice P10": ice[list(FACTORS)].quantile(0.1).values,
    "Ice 중앙값": ice[list(FACTORS)].median().values,
    "Ice P90": ice[list(FACTORS)].quantile(0.9).values,
    "비결빙 중앙값": non[list(FACTORS)].median().values,
    "노면 0°C 이하 비결빙 중앙값": non.loc[non["rst"] <= 0, list(FACTORS)].median().values,
}, index=list(FACTORS))
auc = {}
for col in FACTORS:
    # 단일 변수로 Ice/비결빙을 가르는 힘 (AUC, 0.5=무관). 방향은 Ice 쪽이 낮으면 작을수록 위험
    x = tr[col]
    r = x.rank()
    n1, n0 = len(ice), len(non)
    a = (r[tr["ice"] == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc[col] = max(a, 1 - a)
cond_table["판별력(AUC)"] = pd.Series(auc)
cond_table["위험 방향"] = np.where(cond_table["Ice 중앙값"] < cond_table["비결빙 중앙값"], "낮을수록", "높을수록")
cond_table.sort_values("판별력(AUC)", ascending=False).round(2)

,변수,Ice P10,Ice 중앙값,Ice P90,비결빙 중앙값,노면 0°C 이하 비결빙 중앙값,판별력(AUC),위험 방향
era5_sf3h,"ERA5 3시간 강설(mm, 물 환산)",0.00,0.18,1.26,0.00,0.00,0.87,높을수록
hrs_since_rain,강수 후 경과시간(h),0.00,0.50,40.03,43.50,35.33,0.85,낮을수록
snow,적설(cm),0.00,2.43,13.00,0.00,0.00,0.85,높을수록
era5_tp3h,ERA5 3시간 강수(mm),0.00,0.20,1.39,0.00,0.00,0.83,높을수록
rain_hrs12h,최근 12시간 강수시간(h),0.00,1.89,5.20,0.00,0.00,0.83,높을수록
rst,노면온도(°C),-5.60,-1.70,1.02,3.40,-2.80,0.81,낮을수록
era5_snowfrac3h,ERA5 강설 비율(3시간),0.00,0.96,1.00,0.00,0.00,0.81,높을수록
rst_td,노면온도-이슬점(°C),0.21,2.70,6.62,6.03,5.30,0.78,낮을수록
cloud,전운량(1/10),2.50,9.19,10.00,4.96,4.29,0.77,높을수록
ta,기온(°C),-7.28,-2.20,0.64,1.60,-4.19,0.75,낮을수록


- 노면온도·기온이 낮고, 습도·운량이 높고, 최근 강수·적설이 있을 때 Ice가 나온다.
- 노면온도가 0°C 이하여도 비결빙인 경우(오른쪽 열)와 비교하면, **수분 공급(강수·적설·높은 습도)**이 결빙을 가르는 핵심이다.
- 추가 자료인 **ERA5 3시간 강설**(물 환산)은 단일 변수 판별력이 가장 높다 (AUC 0.87, 다음은 강수 후 경과시간·적설 0.85).
  Ice 사례의 강설 비율 중앙값은 0.96으로, Ice의 절반 이상이 지난 3시간 ERA5 강수 0.1mm 이상이고 그 대부분이 눈이다.
  ERA5 3시간 강수(0.83)는 AWS 최근 12시간 강수시간(0.83)과 같은 수준이다. 비보다 **눈**이 결빙을 더 잘 가른다.

Ice 사례 분포로 임계값을 정한다: 노면온도는 Ice 사례의 90%가 들어가는 상한(P90), 습도는 하한(P10).
수분 조건은 최근 12시간 강수, 적설, 또는 노면온도 ≤ 이슬점(서리·응결) 중 하나이다.

In [9]:
T_RST = float(np.round(ice["rst"].quantile(0.9) * 2) / 2)      # 0.5°C 단위
T_RH = float(np.floor(ice["rh"].quantile(0.1) / 5) * 5)       # 5% 단위


def icing_condition(df, rst_col="rst"):
    """결빙 가능 환경조건: 저온 + 고습 + 수분 공급."""
    moist = (df["rain_hrs12h"] > 0) | (df["snow"] > 0) | ((df[rst_col] - df["td"]) <= 0)
    return (df[rst_col] <= T_RST) & (df["rh"] >= T_RH) & moist


base_rate = tr["ice"].mean()
rows = []
for name, c in {
    f"노면온도 ≤ {T_RST}°C": tr["rst"] <= T_RST,
    f"+ 습도 ≥ {T_RH:.0f}%": (tr["rst"] <= T_RST) & (tr["rh"] >= T_RH),
    "+ 수분 공급 (= 결빙 가능 조건)": icing_condition(tr),
}.items():
    rows.append({"조건": name, "Ice 포착률": c[tr["ice"] == 1].mean(), "해당 시간 비율": c.mean(),
                 "조건 내 Ice 비율": tr.loc[c, "ice"].mean(), "향상도(lift)": tr.loc[c, "ice"].mean() / base_rate})
conditions = pd.DataFrame(rows).set_index("조건")
print(f"전체 Ice 비율 {base_rate:.5f}")
conditions.round(4)

전체 Ice 비율 0.00050


,Ice 포착률,해당 시간 비율,조건 내 Ice 비율,향상도(lift)
조건,,,,
노면온도 ≤ 1.0°C,0.8996,0.3533,0.0013,2.5465
+ 습도 ≥ 70%,0.8285,0.2456,0.0017,3.3728
+ 수분 공급 (= 결빙 가능 조건),0.7531,0.1279,0.0029,5.8873


결빙 가능 조건은 전체 시간의 일부만 차지하면서 Ice 사례의 대부분을 포함하고, 조건 안에서 Ice가 나올 확률은 평균보다 몇 배 높다.

- 자료로 정한 노면온도 임계값(Ice 사례의 90% 상한, 1.0°C)은 Bonanno et al.(2010)의 "강수가 있을 때 노면 1°C 미만에서 결빙" 기준과 같다
  (Hong et al. 2022에서 재인용). 강수가 없을 때의 0°C 기준은 쓰지 않았다. 이 자료에서 강수 없는 Ice의 38%가 0~1.5°C에서 나왔기 때문이다 (01-6의 13-1).
- 수분 공급의 세 조건(최근 강수, 적설, 노면 ≤ 이슬점)은 위 세 경로에 대응한다. 서리 조건(노면 ≤ 이슬점)은 Jang(2024)의 기준이지만,
  이 자료에서는 서리 조건일 때 Ice로 판정된 경우가 드물어(01-6의 13-3) 보조 조건으로만 쓴다.

조건의 조합을 데이터로 확인하려고 얕은 결정나무(깊이 3)도 학습한다.

In [10]:
tree_vars = ["rst", "rh", "rain_hrs12h", "snow", "rst_td", "cloud", "ws"]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, class_weight="balanced", random_state=0)
tree.fit(tr[tree_vars].fillna(0), tr["ice"].astype(int))
print(export_text(tree, feature_names=[FACTORS.get(v, v) for v in tree_vars], decimals=1, show_weights=False))

|--- 적설(cm) <= 0.1
|   |--- 노면온도(°C) <= 1.6
|   |   |--- 상대습도(%) <= 88.6
|   |   |   |--- class: 0
|   |   |--- 상대습도(%) >  88.6
|   |   |   |--- class: 1
|   |--- 노면온도(°C) >  1.6
|   |   |--- 적설(cm) <= 0.0
|   |   |   |--- class: 0
|   |   |--- 적설(cm) >  0.0
|   |   |   |--- class: 0
|--- 적설(cm) >  0.1
|   |--- 최근 12시간 강수시간(h) <= 0.4
|   |   |--- 노면온도-이슬점(°C) <= 3.1
|   |   |   |--- class: 1
|   |   |--- 노면온도-이슬점(°C) >  3.1
|   |   |   |--- class: 0
|   |--- 최근 12시간 강수시간(h) >  0.4
|   |   |--- 노면온도(°C) <= 1.6
|   |   |   |--- class: 1
|   |   |--- 노면온도(°C) >  1.6
|   |   |   |--- class: 0



결정나무는 적설·노면온도·습도·최근 강수·노면-이슬점 차로 나눈다. **저온과 수분 공급의 조합**으로 결빙을 가른다는 점에서
위의 결빙 가능 조건과 같은 방향이다.

관측소 단위로 도로·지형 요인과 결빙 관련 지표의 관계를 본다 (학습 지점 14곳, 순위상관).
지점 수가 적어 통계적 확신은 어렵지만 요인의 방향을 확인하는 근거로 쓴다.

기상 요인 후보로 **ERA5 강설 중 추정 노면온도가 0°C 이하인 시간 비율**(05-2의 M3와 같은 정의)도 넣고,
같은 지표를 AWS 강수 감지로 만든 것과 비교한다. 구간에는 실측 노면온도가 없으므로 둘 다 03의 추정 노면온도(학습 지점은 교차검증 예측)를 쓴다.

In [11]:
tr["era5_frz_snow"] = (tr["era5_sf_mmh"] > 0.05) & (tr["rst_est"] <= 0)  # ERA5 강설 0.05 mm/h 이상 & 추정 노면 0°C 이하
tr["aws_frz_rain"] = (tr["rain"] >= 0.5) & (tr["rst_est"] <= 0)         # 같은 지표를 AWS 강수 감지로
stn_stats = tr.groupby("stn").agg(
    Ice비율=("ice", "mean"),
    결빙관련비율=("state", lambda s: s.isin(cls_meta["label_states"]).mean()),
    노면0도이하비율=("rst", lambda s: (s <= 0).mean()),
    결빙가능조건비율=("rst", lambda s: icing_condition(tr.loc[s.index]).mean()),
    ERA5강설영하노면=("era5_frz_snow", "mean"),
    AWS강수영하노면=("aws_frz_rain", "mean"),
).join(stations[["name", "elev", "tpi", "shade_winter", "dist_bridge", "dist_river"]])
targets = ["Ice비율", "결빙관련비율", "노면0도이하비율", "결빙가능조건비율"]
STN_FACTORS = {"elev": "표고", "tpi": "TPI(지형 상대고도)", "shade_winter": "겨울 지형 그늘", "dist_bridge": "교량까지 거리",
               "dist_river": "하천까지 거리", "ERA5강설영하노면": "ERA5 강설 & 추정 노면 ≤ 0°C",
               "AWS강수영하노면": "AWS 강수 감지 & 추정 노면 ≤ 0°C"}
corr = pd.DataFrame({t: {f: spearmanr(stn_stats[f], stn_stats[t])[0] for f in STN_FACTORS} for t in targets})
corr.index = list(STN_FACTORS.values())
corr.round(2)

,Ice비율,결빙관련비율,노면0도이하비율,결빙가능조건비율
표고,-0.19,-0.21,0.68,0.72
TPI(지형 상대고도),0.27,0.02,-0.48,-0.59
겨울 지형 그늘,-0.34,0.21,0.44,0.48
교량까지 거리,0.38,0.48,-0.53,-0.17
하천까지 거리,-0.02,0.64,-0.53,-0.49
ERA5 강설 & 추정 노면 ≤ 0°C,0.31,0.49,0.14,0.27
AWS 강수 감지 & 추정 노면 ≤ 0°C,0.16,0.45,0.34,0.60


- **노면 0°C 이하 비율**(저온 노출)은 표고·지형 그늘과 양(+)의 관계, 교량·하천과의 거리와 음(-)의 관계이다.
  즉 높고, 그늘지고, 교량·하천에 가까울수록 노면이 자주 얼 만큼 차가워진다.
- Ice 비율 자체는 표고와 관계가 약하다. 서해안선 저지대에서 Ice가 많은데, 강수·적설(수분)이 잦기 때문으로 보인다.
  그래서 위험도는 **기상(수분+저온) 요인**과 **도로·지형(저온 노출) 요인**을 함께 보도록 구성한다.
- **ERA5 강설 & 추정 노면 ≤ 0°C 비율**은 관측소의 Ice 비율과 0.31, 결빙관련 비율과 0.49의 순위상관을 보인다.
  같은 지표를 AWS 강수 감지로 만들면 0.16, 0.45로 조금 낮다. 강수 유무보다 눈인지가 결빙 빈도와 더 관련 있다는 방향이지만,
  지점이 14곳뿐이라(5% 유의 수준에 |ρ| ≥ 0.54 필요) 차이를 확신할 수는 없다.
  05-2에서는 구간이 쓰는 ERA5 격자를 모두 받았을 때만 이 지표(M3)를 기상 요인에 더한다.

In [12]:
stn_stats.round(4)

,Ice비율,결빙관련비율,노면0도이하비율,결빙가능조건비율,ERA5강설영하노면,AWS강수영하노면,name,elev,tpi,shade_winter,dist_bridge,dist_river
stn,,,,,,,,,,,,
12103,0.0000,0.0189,0.2588,0.0867,0.0172,0.0078,군산휴(도),22.180799,-6.549400,0.1828,396.9076,449.6408
12209,0.0005,0.0137,0.3086,0.1125,0.0262,0.0134,서김제(도),20.164400,-0.000000,0.0000,29.8045,436.3778
12210,0.0034,0.0144,0.2381,0.0917,0.0360,0.0102,부안고려청자휴(도),20.164400,-0.038500,0.0015,534.6397,345.2879
12211,0.0001,0.0145,0.2936,0.1032,0.0352,0.0092,고인돌휴(도),60.493099,-35.660301,0.0306,977.9577,744.4217
12409,0.0002,0.0063,0.3174,0.1246,0.0162,0.0068,우곡3교(도),20.164400,-0.423500,0.0051,76.5769,212.9384
23202,0.0000,0.0030,0.2473,0.0982,0.0136,0.0057,여산휴(도),61.501301,-15.580800,0.0246,137.6151,158.5416
23203,0.0001,0.0074,0.2537,0.0892,0.0135,0.0071,삼례졸음(도),20.164400,-0.000000,0.0000,489.2331,482.5026
23204,0.0000,0.0145,0.2404,0.1216,0.0222,0.0119,정읍휴(도),20.164400,-3.510900,0.0055,1064.7175,964.2688
23403,0.0014,0.0222,0.2553,0.1361,0.0290,0.0190,호남터널(도),110.903999,-59.073799,0.0936,3776.6985,380.1644


## 저장

- `data/processed/segment_estimates.parquet`: 구간별 1시간 간격 추정 노면온도·결빙관련 확률 → 05-2
- `data/processed/segment_summary.parquet`: 구간별 노면온도 요약 → 05-2
- `data/processed/icing_conditions.json`: 결빙 가능 조건의 임계값, 조건별 포착률·향상도, 관측소 단위 순위상관(ERA5 강설 지표 포함) → 05-2
- `data/results/icing_factors.csv`, `icing_conditions.csv`: 결빙 영향인자와 환경조건(임계값)

In [13]:
cond_table.round(3).to_csv(config.RESULT_DIR / "icing_factors.csv", encoding="utf-8-sig")
conditions.round(4).to_csv(config.RESULT_DIR / "icing_conditions.csv", encoding="utf-8-sig")

seg_feat[["seg_id", "time", "rst_est", "rst_dpd", "p_icing"]].to_parquet(
    config.PROCESSED_DIR / "segment_estimates.parquet", index=False)
seg_summary.to_parquet(config.PROCESSED_DIR / "segment_summary.parquet")
with open(config.PROCESSED_DIR / "icing_conditions.json", "w", encoding="utf-8") as f:
    json.dump({"T_RST": T_RST, "T_RH": T_RH, "conditions": conditions.to_dict(orient="index"),
               "station_corr": corr.to_dict()}, f, ensure_ascii=False, indent=2)
print("저장 완료")

저장 완료
